# Ingestion Pipeline Experiments — docs → pgvector

Experimentation notebook for **module 1.4** (`app/retrieval/ingestion.py`). The production
pipeline is a **separate offline job** — it is not part of the request graph. Use this
notebook to settle the ingestion decisions (loaders, chunking parameters, embedding model,
collection writes), then port the final choices into `app/retrieval/ingestion.py` +
`app/config/settings.py` and verify via the Stage 1 cells of `v1_module_tests.ipynb`.

**Design constraints (from the v1 reference, do not deviate):**
- Vector store: **pgvector** (reuses existing PostgreSQL — no new infra)
- **One collection per product**, names from the registry (`ProductConfig.doc_collection`)
- Chunks carry **source metadata** (needed for `[D1]`-style citations downstream)
- Registry-driven: no hardcoded product names

**LangChain components used:** document loaders (`PyPDFLoader` / `Docx2txtLoader` /
`TextLoader`), `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `PGVector`
(`langchain_postgres`).

---
## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set"
PG_CONN = os.environ["DATABASE_URL"]    # postgresql+psycopg://... (see .env)
print("Repo root:", REPO_ROOT)

In [ ]:
# Registry drives everything: products and their collection names
from app.core.registry import Registry

registry = Registry(path=str(REPO_ROOT / "app/config/products.yaml"))
for pid in registry.product_ids():
    print(pid, "->", registry.get(pid).doc_collection)

---
## 1. Document inventory

Convention: raw product documentation lives at `data/docs/<product_id>/`, one folder per
product, matching registry IDs. Supported formats: `.pdf`, `.docx`, `.md`, `.txt`.

In [ ]:
DOCS_ROOT = REPO_ROOT / "data" / "docs"

inventory = {}
for pid in registry.product_ids():
    folder = DOCS_ROOT / pid
    files = sorted(p for p in folder.glob("*") if p.suffix.lower() in {".pdf", ".docx", ".md", ".txt"}) \
            if folder.exists() else []
    inventory[pid] = files
    print(f"{pid}: {len(files)} file(s)")
    for f in files:
        print("   ", f.name)

missing = [pid for pid, fs in inventory.items() if not fs]
if missing:
    print("\nWARNING — no docs found for:", missing)

---
## 2. Loading — LangChain document loaders

One loader per file type. Every loaded `Document` gets `product` and `source` metadata —
`source` is what the retriever surfaces and what citations ultimately point back to.

**Chunk-metadata contract — two groups, stamped at different stages:**

*Provenance* (stamped here, at load time — describes the source file):
`product`, `source`, `doc_version`, `ingested_at`, `content_hash`.
Purpose: answer → exact document version audit trail.

*Build config* (stamped in §4, at chunk time — describes how the vector was produced):
`embedding_model`, `chunk_size`, `chunk_overlap`.
Purpose: let the retriever detect that `settings.py` has drifted from what the store was
actually built with. An `embedding_model` mismatch is otherwise **silent** — query vectors
from model B compared against document vectors from model A return arbitrary chunks with
no error, and every downstream gate (`verify` included) still passes.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
import hashlib, re
from datetime import date

LOADERS = {
    ".pdf":  PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".md":   TextLoader,
    ".txt":  TextLoader,
}

# Provenance half of the contract — stamped at load time, describes the source file.
# The build-config half (embedding_model, chunk_size, chunk_overlap) is stamped in §4.
PROVENANCE_META = {"product", "source", "doc_version", "ingested_at", "content_hash"}

def parse_last_updated(text: str) -> str:
    """Extract the doc's 'Last updated: <Month Year>' header; fall back to 'unknown'."""
    m = re.search(r"Last updated:\**\s*([A-Za-z]+\s+\d{4})", text)
    return m.group(1) if m else "unknown"

def load_product_docs(product_id: str):
    """Load docs and stamp the provenance metadata:
    product, source, doc_version, ingested_at, content_hash.
    Format-agnostic: works identically for .docx, .pdf, .md, .txt because
    doc_version is parsed from the LOADED text, not the raw file bytes."""
    docs = []
    for path in inventory[product_id]:
        content_hash = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
        loader = LOADERS[path.suffix.lower()](str(path))
        loaded = loader.load()
        # Parse 'Last updated: <Month Year>' from the extracted text — the header just
        # needs to exist in the document body, whatever the file format (Word included).
        full_text = "\n".join(d.page_content for d in loaded)
        doc_version = parse_last_updated(full_text)
        for d in loaded:
            d.metadata["product"] = product_id
            d.metadata["source"] = path.name
            d.metadata["doc_version"] = doc_version           # e.g. "July 2026"
            d.metadata["ingested_at"] = date.today().isoformat()
            d.metadata["content_hash"] = content_hash          # ties chunks to exact file version
            docs.append(d)
    return docs

# Smoke test on one product
sample_pid = registry.product_ids()[0]
raw_docs = load_product_docs(sample_pid)
print(f"{sample_pid}: {len(raw_docs)} raw document(s)/page(s)")
print("\nFirst 400 chars of first doc:\n", raw_docs[0].page_content[:400])
print("\nmetadata:", raw_docs[0].metadata)

# Provenance contract check — every doc must carry all five fields
for d in raw_docs:
    missing = PROVENANCE_META - d.metadata.keys()
    assert not missing, f"Missing metadata fields: {missing}"
print("Provenance metadata OK:", sorted(PROVENANCE_META))

# WATCH THIS: 'unknown' means the 'Last updated:' header did not survive text extraction.
# It is not an error here, but it would render as "updated unknown" in every citation
# downstream (respond, Stage 2). Fix the doc or the regex before porting to ingestion.py.
unresolved = {d.metadata["source"] for d in raw_docs if d.metadata["doc_version"] == "unknown"}
if unresolved:
    print("\n!! doc_version unresolved for:", sorted(unresolved))
else:
    print("doc_version parsed:", {d.metadata["source"]: d.metadata["doc_version"] for d in raw_docs})

---
## 3. Chunking experiments — `RecursiveCharacterTextSplitter`

The parameter that most affects retrieval quality. Compare a few configs on real docs,
eyeball the chunks, and pick ONE config to freeze into `settings.py`. Judgment criteria:
chunks should be self-contained enough to answer a question alone (they become the `[D1]`
context units), without being so large that top-k retrieval drags in noise.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CONFIGS = [
    {"chunk_size": 500,  "chunk_overlap": 50},
    {"chunk_size": 1000, "chunk_overlap": 150},
    {"chunk_size": 1500, "chunk_overlap": 200},
]

for cfg in CONFIGS:
    splitter = RecursiveCharacterTextSplitter(**cfg)
    chunks = splitter.split_documents(raw_docs)
    lens = [len(c.page_content) for c in chunks]
    print(f"size={cfg['chunk_size']:>4} overlap={cfg['chunk_overlap']:>3} -> "
          f"{len(chunks):>4} chunks | avg len {sum(lens)//max(len(lens),1)}")

In [ ]:
# Inspect actual chunks for one config before deciding
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(raw_docs)

for c in chunks[:3]:
    print("=" * 80)
    print("source:", c.metadata.get("source"), "| product:", c.metadata.get("product"))
    print(c.page_content[:500])

In [ ]:
# FREEZE the decision here once you've compared — this is what goes into settings.py
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"Frozen: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")

---
## 4. Embeddings + pgvector collections

`PGVector` from `langchain_postgres`, one collection per product (name from the registry).
`pre_delete_collection=True` makes a full re-ingest idempotent — re-running replaces the
collection instead of duplicating chunks.

**Build-config stamping (the other half of the metadata contract).** Every chunk also
records `embedding_model`, `chunk_size`, `chunk_overlap` — the settings that produced its
vector. `retriever.py` (module 1.5) reads one chunk per collection at startup and compares
these against `settings.py`, so a config change after ingestion fails loudly instead of
silently degrading retrieval:

- `embedding_model` mismatch → **raise**. Vectors from two different models are not
  comparable; if the dimensions happen to match, pgvector returns arbitrary chunks with no
  error at all. Fail closed.
- `chunk_size` / `chunk_overlap` mismatch → **warn**. The store is merely stale, not wrong.

Either way the fix is the same: re-run the offline job.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

EMBEDDING_MODEL = "text-embedding-3-small"   # freeze into settings.py once validated
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity: embed one string
vec = embeddings.embed_query("fixed deposit premature withdrawal penalty")
print("Embedding dims:", len(vec))

In [ ]:
BUILD_CONFIG_META = {"embedding_model", "chunk_size", "chunk_overlap"}

def stamp_build_config(chunks):
    """Record the settings that produced these vectors, so retriever.py can detect
    that settings.py has drifted from what the store was actually built with."""
    for c in chunks:
        c.metadata["embedding_model"] = EMBEDDING_MODEL
        c.metadata["chunk_size"] = CHUNK_SIZE
        c.metadata["chunk_overlap"] = CHUNK_OVERLAP
    return chunks

def ingest_product(product_id: str) -> int:
    """Load -> chunk -> stamp -> embed -> write to the product's collection. Idempotent."""
    cfg = registry.get(product_id)
    docs = load_product_docs(product_id)
    chunks = stamp_build_config(splitter.split_documents(docs))
    PGVector.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=cfg.doc_collection,      # from the registry — never hardcoded
        connection=PG_CONN,
        pre_delete_collection=True,              # full-replace re-ingest
    )
    return len(chunks)

# Ingest ONE product first
n = ingest_product(sample_pid)
print(f"{sample_pid}: {n} chunks written to '{registry.get(sample_pid).doc_collection}'")

In [ ]:
# Ingest ALL products (registry-driven loop — this is the shape of ingestion.py's ingest_all)
for pid in registry.product_ids():
    if not inventory[pid]:
        print(f"SKIP {pid} — no docs")
        continue
    n = ingest_product(pid)
    print(f"OK {pid}: {n} chunks -> {registry.get(pid).doc_collection}")

---
## 5. Retrieval sanity checks

Same checks Stage 1 of `v1_module_tests.ipynb` will run against `app/retrieval/retriever.py` —
run them here first against the raw store to validate the *ingestion*, independent of your
retriever code.

In [ ]:
def open_store(product_id: str) -> PGVector:
    return PGVector(
        embeddings=embeddings,
        collection_name=registry.get(product_id).doc_collection,
        connection=PG_CONN,
    )

sample_queries = {
    "digital_fd":   "What is the penalty for premature withdrawal of a fixed deposit?",
    "digital_gold": "How is the gold price determined at purchase?",
    "bonds":        "What is the minimum investment amount for bonds?",
    "mutual_funds": "How is NAV calculated?",
}

for pid, q in sample_queries.items():
    if not inventory.get(pid):
        continue
    store = open_store(pid)
    results = store.similarity_search_with_score(q, k=4)
    print(f"\n=== {pid} :: {q}")
    for doc, score in results:
        print(f"  score={score:.3f} [{doc.metadata.get('source')}] {doc.page_content[:110]}...")

In [ ]:
# Scoping + full metadata-contract check on RETRIEVED chunks:
# (a) every result in a product's collection carries that product's metadata;
# (b) both halves of the contract survived splitting and storage.
FULL_META = PROVENANCE_META | BUILD_CONFIG_META
for pid in registry.product_ids():
    if not inventory.get(pid):
        continue
    store = open_store(pid)
    results = store.similarity_search("investment", k=8)
    assert all(d.metadata.get("product") == pid for d in results), \
        f"Cross-product contamination in collection {pid}!"
    for d in results:
        missing = FULL_META - d.metadata.keys()
        assert not missing, f"{pid}: chunk lost metadata fields {missing}"
        assert d.metadata["embedding_model"] == EMBEDDING_MODEL, \
            f"{pid}: chunk built with {d.metadata['embedding_model']}, expected {EMBEDDING_MODEL}"
print("OK — collections scoped per product; provenance + build-config intact on every chunk")
print("Sample:", results[0].metadata)

In [ ]:
# Idempotency check: re-ingest one product, chunk count must be stable (no duplication)
n1 = ingest_product(sample_pid)
n2 = ingest_product(sample_pid)
assert n1 == n2, "Re-ingest changed chunk count — pre_delete_collection not working"
store = open_store(sample_pid)
results = store.similarity_search("test", k=200)
print(f"OK — idempotent re-ingest ({n1} chunks both runs)")

In [ ]:
# Config-drift guard — prototype of the check that ports into retriever.py (module 1.5).
# Reads ONE chunk per collection and compares its build config against current settings.
# embedding_model mismatch -> raise (silent corruption); chunk params -> warn (merely stale).
import warnings

class StoreConfigMismatch(RuntimeError):
    """The store was built with different settings than the ones now configured."""

def assert_store_matches(product_id: str, embedding_model: str,
                         chunk_size: int, chunk_overlap: int) -> dict:
    collection = registry.get(product_id).doc_collection
    probe = open_store(product_id).similarity_search("a", k=1)
    if not probe:
        raise StoreConfigMismatch(
            f"collection '{collection}' is empty — run ingestion for '{product_id}'")
    meta = probe[0].metadata
    built_with = meta.get("embedding_model")
    if built_with != embedding_model:
        raise StoreConfigMismatch(
            f"collection '{collection}' was built with embedding model '{built_with}', "
            f"but settings specify '{embedding_model}'. Vectors from different models are "
            f"not comparable — re-run: python -m app.retrieval.ingestion")
    for field, current in (("chunk_size", chunk_size), ("chunk_overlap", chunk_overlap)):
        if meta.get(field) != current:
            warnings.warn(
                f"collection '{collection}' was built with {field}={meta.get(field)}, "
                f"settings say {current}. Store is stale — re-run ingestion.", stacklevel=2)
    return meta

# Happy path: current settings match what we just ingested.
for pid in registry.product_ids():
    if inventory.get(pid):
        assert_store_matches(pid, EMBEDDING_MODEL, CHUNK_SIZE, CHUNK_OVERLAP)
print("OK — every collection matches current settings")

# Negative test: pretend settings.py was switched to -large. Must RAISE, not warn.
try:
    assert_store_matches(sample_pid, "text-embedding-3-large", CHUNK_SIZE, CHUNK_OVERLAP)
except StoreConfigMismatch as exc:
    print("\nOK — drift detected as expected:\n ", exc)
else:
    raise AssertionError("drift guard did NOT fire on an embedding-model mismatch")

---
## 6. Handoff to production (`app/retrieval/ingestion.py`)

Once the cells above pass and you're satisfied with retrieval quality:

1. Freeze into `app/config/settings.py`: `CHUNK_SIZE`, `CHUNK_OVERLAP`, `EMBEDDING_MODEL`,
   `DATABASE_URL` env name, `DOCS_ROOT` convention (`data/docs/<product_id>/`).
   The notebook-local literals above exist only so you can vary them while experimenting —
   they do **not** come along. `ingestion.py` reads every one of them from settings.
1a. Port the **chunk-metadata contract** as-is, both halves:
   - *provenance* — `product`, `source`, `doc_version`, `ingested_at`, `content_hash`.
     Consumers: `retrieve_docs` carries it into `doc_context`; `respond` renders
     "Source: <source>, updated <doc_version>"; observability logs it with every verify
     verdict (answer → exact doc version audit trail).
   - *build config* — `embedding_model`, `chunk_size`, `chunk_overlap`. Consumer:
     `retriever.py`'s startup drift guard (see step 4).
2. Port `load_product_docs`, `stamp_build_config`, the frozen splitter, and
   `ingest_product` / the registry loop into `app/retrieval/ingestion.py` as
   `ingest_all(registry)` — same code, minus the experimentation cells. Expose it as a
   runnable job (`python -m app.retrieval.ingestion`), since a human or a cron invokes it,
   not the app.
3. Run **Stage 1** of `notebooks/v1_module_tests.ipynb` (cells 1.4–1.6) against the
   production module and mark **1.4** ✅ in `docs/module_tracker.md`.
4. Port `assert_store_matches` into `app/retrieval/retriever.py` (module 1.5), cached
   per collection so it costs one probe per process, not one per query.

Re-run ingestion (offline job) whenever product documentation changes — **or whenever
`EMBEDDING_MODEL`, `CHUNK_SIZE`, or `CHUNK_OVERLAP` changes.** The store is built from
those settings; changing them without re-ingesting leaves the store and the config
disagreeing, which is exactly what the drift guard exists to catch.